In [1]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

import albumentations as A
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


In [4]:
#configure
CFG = {
    "data_root": r"C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound",   # change this
    "img_size": 384,
    "batch_size": 4,
    "epochs": 40,
    "lr": 1e-3,
    "num_workers": 0,   # Windows/Jupyter safest
    "threshold": 0.5,
}

In [5]:
#inspect datasets
DATASET_ROOT = Path(CFG["data_root"])

print("Root exists:", DATASET_ROOT.exists())
print("\nTop-level items:")
for p in sorted(DATASET_ROOT.iterdir()):
    print("-", p.name)

csv_files = list(DATASET_ROOT.rglob("*.csv"))
png_files = list(DATASET_ROOT.rglob("*.png"))

print("\nCSV files:")
for p in csv_files:
    print(p)

print("\nTotal PNG files:", len(png_files))
print("Example PNGs:")
for p in png_files[:10]:
    print(p)

Root exists: True

Top-level items:
- archive
- BUSBRA
- busbra_birads
- Dataset_BUSI_with_GT
- venv

CSV files:
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\bus_data.csv

Total PNG files: 7203
Example PNGs:
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0001-l.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0001-r.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0002-l.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0002-r.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0003-l.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0003-r.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0004-l.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\Images\bus_0004-r.png
C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\I

In [6]:
#Preview csv column
for csv_path in csv_files:
    print("\n======================")
    print("CSV:", csv_path)
    tmp = pd.read_csv(csv_path)
    print("Columns:", tmp.columns.tolist())
    display(tmp.head())


CSV: C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\bus_data.csv
Columns: ['ID', 'Case', 'Histology', 'Pathology', 'BIRADS', 'Device', 'Width', 'Height', 'Side', 'HOB', 'K5B', 'K10B', 'HOP', 'K5P', 'K10P', 'BBOX']


,ID,Case,Histology,Pathology,BIRADS,Device,Width,Height,Side,HOB,K5B,K10B,HOP,K5P,K10P,BBOX
0,bus_0001-l,1,invasive ductal carcinoma,malignant,4,GE Logiq 7 @10-14MHz,274,353,left,2,3,3,1,3,9,"[91,24,103,79]"
1,bus_0001-r,1,invasive ductal carcinoma,malignant,4,GE Logiq 7 @10-14MHz,275,353,right,2,3,3,1,3,9,"[102,24,82,79]"
2,bus_0002-l,2,fibroadenoma,benign,4,GE Logiq 7 @10-14MHz,274,371,left,2,2,4,2,5,3,"[134,142,88,50]"
3,bus_0002-r,2,fibroadenoma,benign,4,GE Logiq 7 @10-14MHz,276,371,right,2,2,4,2,5,3,"[113,143,68,47]"
4,bus_0003-l,3,invasive ductal carcinoma,malignant,4,GE Logiq 7 @10-14MHz,275,372,left,1,4,9,2,4,7,"[71,78,126,137]"


In [12]:
#metadata frame
META_CSV = DATASET_ROOT / "metadata.csv"   # change this
meta = pd.read_csv(META_CSV)

IMAGES_DIR = DATASET_ROOT / "images"       # change if needed
MASKS_DIR  = DATASET_ROOT / "masks"        # change if needed

df = pd.DataFrame({
    "image_path": meta["image_name"].apply(lambda x: str(IMAGES_DIR / x)),
    "mask_path":  meta["mask_name"].apply(lambda x: str(MASKS_DIR / x)),
    "patient_id": meta["patient_id"],
    "label":      meta["pathology"],   # optional for later classification
})

print(df.head())
print(df.columns.tolist())
print("Total:", len(df))
print(df["label"].value_counts(dropna=False))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ASHAH\\Desktop\\DL_Project\\breast_ultrasound\\metadata.csv'

In [8]:
from pathlib import Path

DATASET_ROOT = Path(r"C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound")

csv_files = list(DATASET_ROOT.rglob("*.csv"))

print("CSV count:", len(csv_files))
for i, p in enumerate(csv_files):
    print(i, p)

CSV count: 1
0 C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\bus_data.csv


In [9]:
import pandas as pd

for i, p in enumerate(csv_files):
    print("\n======================")
    print("Index:", i)
    print("Path :", p)
    tmp = pd.read_csv(p)
    print("Columns:", tmp.columns.tolist())
    print(tmp.head())


Index: 0
Path : C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\bus_data.csv
Columns: ['ID', 'Case', 'Histology', 'Pathology', 'BIRADS', 'Device', 'Width', 'Height', 'Side', 'HOB', 'K5B', 'K10B', 'HOP', 'K5P', 'K10P', 'BBOX']
           ID  Case                  Histology  Pathology  BIRADS  \
0  bus_0001-l     1  invasive ductal carcinoma  malignant       4   
1  bus_0001-r     1  invasive ductal carcinoma  malignant       4   
2  bus_0002-l     2               fibroadenoma     benign       4   
3  bus_0002-r     2               fibroadenoma     benign       4   
4  bus_0003-l     3  invasive ductal carcinoma  malignant       4   

                 Device  Width  Height   Side  HOB  K5B  K10B  HOP  K5P  K10P  \
0  GE Logiq 7 @10-14MHz    274     353   left    2    3     3    1    3     9   
1  GE Logiq 7 @10-14MHz    275     353  right    2    3     3    1    3     9   
2  GE Logiq 7 @10-14MHz    274     371   left    2    2     4    2    5     3   
3  GE Logiq 7 @1

In [13]:
META_CSV = csv_files[0]   # change index if needed
meta = pd.read_csv(META_CSV)

print(META_CSV)
print(meta.columns.tolist())
print(meta.head())

C:\Users\ASHAH\Desktop\DL_Project\breast_ultrasound\BUSBRA\BUSBRA\bus_data.csv
['ID', 'Case', 'Histology', 'Pathology', 'BIRADS', 'Device', 'Width', 'Height', 'Side', 'HOB', 'K5B', 'K10B', 'HOP', 'K5P', 'K10P', 'BBOX']
           ID  Case                  Histology  Pathology  BIRADS  \
0  bus_0001-l     1  invasive ductal carcinoma  malignant       4   
1  bus_0001-r     1  invasive ductal carcinoma  malignant       4   
2  bus_0002-l     2               fibroadenoma     benign       4   
3  bus_0002-r     2               fibroadenoma     benign       4   
4  bus_0003-l     3  invasive ductal carcinoma  malignant       4   

                 Device  Width  Height   Side  HOB  K5B  K10B  HOP  K5P  K10P  \
0  GE Logiq 7 @10-14MHz    274     353   left    2    3     3    1    3     9   
1  GE Logiq 7 @10-14MHz    275     353  right    2    3     3    1    3     9   
2  GE Logiq 7 @10-14MHz    274     371   left    2    2     4    2    5     3   
3  GE Logiq 7 @10-14MHz    276     371  ri

In [15]:
META_CSV = DATASET_ROOT / "metadata.csv"

In [17]:
print(meta.columns.tolist())
print(meta.head())

['ID', 'Case', 'Histology', 'Pathology', 'BIRADS', 'Device', 'Width', 'Height', 'Side', 'HOB', 'K5B', 'K10B', 'HOP', 'K5P', 'K10P', 'BBOX']
           ID  Case                  Histology  Pathology  BIRADS  \
0  bus_0001-l     1  invasive ductal carcinoma  malignant       4   
1  bus_0001-r     1  invasive ductal carcinoma  malignant       4   
2  bus_0002-l     2               fibroadenoma     benign       4   
3  bus_0002-r     2               fibroadenoma     benign       4   
4  bus_0003-l     3  invasive ductal carcinoma  malignant       4   

                 Device  Width  Height   Side  HOB  K5B  K10B  HOP  K5P  K10P  \
0  GE Logiq 7 @10-14MHz    274     353   left    2    3     3    1    3     9   
1  GE Logiq 7 @10-14MHz    275     353  right    2    3     3    1    3     9   
2  GE Logiq 7 @10-14MHz    274     371   left    2    2     4    2    5     3   
3  GE Logiq 7 @10-14MHz    276     371  right    2    2     4    2    5     3   
4  GE Logiq 7 @10-14MHz    275     372  

In [18]:
IMAGES_DIR = DATASET_ROOT / "images"
MASKS_DIR = DATASET_ROOT / "masks"

df = pd.DataFrame({
    "image_path": meta["image_name"].apply(lambda x: str(IMAGES_DIR / x)),
    "mask_path": meta["mask_name"].apply(lambda x: str(MASKS_DIR / x)),
    "patient_id": meta["patient_id"],
    "label": meta["pathology"],
})

print(df.head())
print(df.columns.tolist())
print(len(df))

KeyError: 'image_name'